# Oracle-style comparison: OAMP vs EBR on the *preservation-after-correction* assertion

**Spec origin.** Oracle DevRel's `oracle_agent_memory_benchmarks.ipynb` runs an 80-turn dialogue scenario with three apples-to-apples comparisons (tokens / latency / LLM-judge quality) and honestly documents where their own OAMP product underperforms. This notebook adopts the same benchmark discipline but adds **one EBR-specific assertion** Oracle's benchmark cannot fail or pass by design:

> *After a fact F is asserted at turn N and contradicted by an amendment at turn M (M > N), the substrate must (a) return the amendment as the dominant retrieval at turn 80, AND (b) preserve F as a retrievable historical record — never delete it.*

OAMP's `delete_memory` semantics make (b) impossible by construction. EBR's append-never-mutate semantics make (b) free, and structural precedence makes (a) deterministic rather than probabilistic.

**Status.** Spec + scaffold notebook. Real substrate calls are stubbed with mock implementations that capture the contract; the harness, scenario, and assertions are runnable as-is. Operator + co-authors fill in the real-substrate adapters when accretive-substrate is public.

**This notebook is the empirical anchor for the LNCS §5 HIPAA argument and the position paper §6 EBR↔Verum mapping.**

## 1. Experiment design

### 1.1 Scenario — 80-turn medical-dialogue compatible with both substrates

A clinical agent runs an 80-turn dialogue with a patient. At turn N=12, the agent records:

> *Fact F: patient is allergic to penicillin (source: patient self-report).*

At turn M=47, an updated lab result amends F:

> *Amendment A: patient is NOT allergic to penicillin (source: 2026-05 lab panel; prior self-report was misattributed reaction to amoxicillin in 2019).*

At turn 80, a downstream system queries the substrate for *patient's penicillin allergy status*. Three assertions are evaluated:

| # | Assertion | OAMP expected | EBR expected |
|---|-----------|---------------|--------------|
| A1 | **Amendment dominates retrieval** — the response cites A as current truth | depends on cosine ranking + recency heuristics | guaranteed by structural-precedence invariant |
| A2 | **Original preserved** — F is still retrievable when querying for *history of patient's allergy claims* | impossible if `delete_memory` was called; otherwise dependent on extractor policy | guaranteed by append-never-mutate |
| A3 | **Audit trail complete** — full chain (F → A) reconstructable with timestamps, authors, source citations | requires separate audit log (Audit Vault add-on) | guaranteed by append-only + Verum hash chain |

### 1.2 Metrics (matching Oracle's discipline)

- **Tokens consumed** per turn (substrate context injection cost)
- **Latency** for retrieval at turn 80
- **LLM-judge quality** of the response at turn 80 (graded by an external judge model on a 1-5 rubric)
- **Assertion correctness** (A1/A2/A3) — binary per assertion, per substrate

## 2. Setup

In [ ]:
from __future__ import annotations

import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional

# Reproducibility
TURN_F = 12      # turn at which fact F is asserted
TURN_M = 47      # turn at which amendment A overrides F
TOTAL_TURNS = 80
FINAL_QUERY_TURN = 80

## 3. Substrate record model — minimum viable for both

Both substrates store records with:
- `id` — unique identifier
- `subject` — the entity the record is about (e.g., `patient/allergy`)
- `claim` — the textual claim
- `author` — operator | extractor | system
- `created_at` — ISO timestamp
- `source_citation` — provenance

OAMP additionally allows the record to be **deleted**. EBR additionally requires:
- `supersedes` — optional prior record id this amendment dominates (structural precedence)
- `provenance_class` — `operator_authored_realtime` | `journal_distilled_backfill` | `coach_provisional`

In [ ]:
@dataclass
class Record:
    id: str
    subject: str
    claim: str
    author: str
    created_at: str
    source_citation: str
    # EBR-only fields
    supersedes: Optional[str] = None
    provenance_class: Optional[str] = None
    # OAMP-internal
    deleted: bool = False

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def new_id() -> str:
    return str(uuid.uuid4())[:8]

## 4. OAMP substrate — mock implementing the Oracle Memory product semantics

Two operations against the canonical OAMP shape:
- `add_memory(record)` — append
- `delete_memory(record_id)` — remove
- `query(subject)` — cosine-ranked retrieval over current (non-deleted) records

The Oracle DevRel notebook explicitly demonstrates `delete_memory` for *correction* workflows. We model that exactly.

In [ ]:
class OAMPSubstrate:
    """Mock of Oracle Agent Memory Platform semantics — delete-then-add for corrections."""
    def __init__(self):
        self.records: dict[str, Record] = {}
        self.deleted_log: list[str] = []  # OAMP does NOT preserve deletes — only an external Audit Vault would

    def add_memory(self, r: Record) -> None:
        self.records[r.id] = r

    def delete_memory(self, record_id: str) -> None:
        if record_id in self.records:
            self.records[record_id].deleted = True
            self.deleted_log.append(record_id)

    def correct(self, old_id: str, new_record: Record) -> None:
        """Canonical OAMP correction workflow per Oracle DevRel notebook."""
        self.delete_memory(old_id)
        self.add_memory(new_record)

    def query(self, subject: str) -> list[Record]:
        """Cosine-ranked retrieval; here a simple subject-prefix match for the mock."""
        return [r for r in self.records.values() if r.subject == subject and not r.deleted]

    def query_history(self, subject: str) -> list[Record]:
        """Historical retrieval — only returns what is still in the store (deleted are gone)."""
        return [r for r in self.records.values() if r.subject == subject and not r.deleted]

## 5. EBR substrate — mock implementing accretive-substrate semantics

Two operations against the EBR shape:
- `accrete(record)` — append-only
- `amend(old_id, new_record_with_supersedes_set)` — append (NEVER mutates / deletes the old record)
- `query(subject)` — returns the head of the supersession chain (structural precedence)
- `query_history(subject)` — returns the full chain including superseded records

In [ ]:
class EBRSubstrate:
    """Mock of Evidence-Bound Retrieval — append-never-mutate with structural precedence."""
    def __init__(self):
        self.records: list[Record] = []

    def accrete(self, r: Record) -> None:
        self.records.append(r)

    def amend(self, old_id: str, new_record: Record) -> None:
        assert new_record.supersedes == old_id, "EBR amendment MUST cite the record it supersedes"
        self.records.append(new_record)

    def query(self, subject: str) -> list[Record]:
        """Return only head-of-chain records (structural precedence invariant — Theorem 1).
        A record is at the head if no later record in the store supersedes it."""
        all_for_subject = [r for r in self.records if r.subject == subject]
        superseded_ids = {r.supersedes for r in all_for_subject if r.supersedes}
        return [r for r in all_for_subject if r.id not in superseded_ids]

    def query_history(self, subject: str) -> list[Record]:
        """Return the FULL supersession chain, including amended-over records."""
        return [r for r in self.records if r.subject == subject]

    def audit_trail(self, head_id: str) -> list[Record]:
        """Reconstruct the chain from head_id back to the original assertion."""
        chain = []
        by_id = {r.id: r for r in self.records}
        cursor: Optional[str] = head_id
        while cursor and cursor in by_id:
            r = by_id[cursor]
            chain.append(r)
            cursor = r.supersedes
        return chain  # head first, original last

## 6. Scenario script — 80-turn dialogue

Each turn writes filler chatter to both substrates so retrieval at turn 80 has realistic noise.
At turn 12, F is asserted. At turn 47, A overrides F (OAMP: delete+add; EBR: amend).

In [ ]:
def run_scenario():
    oamp = OAMPSubstrate()
    ebr = EBRSubstrate()

    # Identifiers held outside the loop so we can amend later
    f_id = new_id()
    a_id = new_id()

    for turn in range(1, TOTAL_TURNS + 1):
        # Generic filler — both substrates receive identical irrelevant chatter
        filler = Record(
            id=new_id(),
            subject=f"chatter/turn-{turn}",
            claim=f"unrelated turn-{turn} dialogue chatter",
            author="operator",
            created_at=now_iso(),
            source_citation=f"turn-{turn}",
            provenance_class="operator_authored_realtime",
        )
        oamp.add_memory(filler)
        ebr.accrete(filler)

        if turn == TURN_F:
            f = Record(
                id=f_id,
                subject="patient/allergy/penicillin",
                claim="Patient is allergic to penicillin.",
                author="operator",
                created_at=now_iso(),
                source_citation="patient self-report 2026-04-12",
                provenance_class="operator_authored_realtime",
            )
            oamp.add_memory(f)
            ebr.accrete(f)

        if turn == TURN_M:
            a = Record(
                id=a_id,
                subject="patient/allergy/penicillin",
                claim="Patient is NOT allergic to penicillin (prior report was misattributed amoxicillin reaction, 2019).",
                author="operator",
                created_at=now_iso(),
                source_citation="lab panel 2026-05-29",
                supersedes=f_id,
                provenance_class="operator_authored_realtime",
            )
            # OAMP canonical correction: delete old, add new
            oamp.correct(old_id=f_id, new_record=a)
            # EBR: amend (append the new with supersedes pointer; never touch old)
            ebr.amend(old_id=f_id, new_record=a)

    return oamp, ebr, f_id, a_id

oamp, ebr, F_ID, A_ID = run_scenario()
print(f"OAMP records: {len(oamp.records)}  (deleted: {len(oamp.deleted_log)})")
print(f"EBR  records: {len(ebr.records)}   (no deletes possible by construction)")

## 7. Run the three assertions

In [ ]:
def assertion_results(oamp: OAMPSubstrate, ebr: EBRSubstrate, f_id: str, a_id: str) -> dict:
    subj = "patient/allergy/penicillin"

    # A1: Amendment dominates retrieval at turn 80
    oamp_head = oamp.query(subj)
    ebr_head = ebr.query(subj)
    a1_oamp = len(oamp_head) == 1 and oamp_head[0].id == a_id
    a1_ebr  = len(ebr_head)  == 1 and ebr_head[0].id  == a_id

    # A2: Original F is preserved in historical retrieval
    oamp_hist = oamp.query_history(subj)
    ebr_hist  = ebr.query_history(subj)
    a2_oamp = any(r.id == f_id for r in oamp_hist)
    a2_ebr  = any(r.id == f_id for r in ebr_hist)

    # A3: Full audit trail (F → A) reconstructable from the substrate alone
    # OAMP: requires external Audit Vault — substrate alone CANNOT reconstruct deleted F.
    # EBR: reconstructable via the supersession pointer chain.
    oamp_audit_chain = oamp.query_history(subj)
    a3_oamp = any(r.id == f_id for r in oamp_audit_chain) and any(r.id == a_id for r in oamp_audit_chain)
    ebr_audit_chain = ebr.audit_trail(a_id)
    a3_ebr  = [r.id for r in ebr_audit_chain] == [a_id, f_id]

    return {
        "A1 amendment dominates": {"OAMP": a1_oamp, "EBR": a1_ebr},
        "A2 original preserved":  {"OAMP": a2_oamp, "EBR": a2_ebr},
        "A3 audit reconstructable": {"OAMP": a3_oamp, "EBR": a3_ebr},
    }

results = assertion_results(oamp, ebr, F_ID, A_ID)
print(json.dumps(results, indent=2))

## 8. Results table

In [ ]:
def render_table(results: dict) -> str:
    rows = ["| Assertion | OAMP | EBR |", "|---|---|---|"]
    for assertion, by_substrate in results.items():
        oamp_mark = "✅" if by_substrate["OAMP"] else "❌"
        ebr_mark  = "✅" if by_substrate["EBR"]  else "❌"
        rows.append(f"| {assertion} | {oamp_mark} | {ebr_mark} |")
    return "\n".join(rows)

print(render_table(results))

## 9. Expected output and what it means

When the scenario runs as specified above, the table should read:

| Assertion | OAMP | EBR |
|---|---|---|
| A1 amendment dominates | ✅ | ✅ |
| A2 original preserved | ❌ | ✅ |
| A3 audit reconstructable | ❌ | ✅ |

**A1 passes for both substrates** — both can return the latest claim as the current one. OAMP achieves this by deleting the prior; EBR by structurally superseding it. Functionally equivalent at the retrieval point.

**A2 fails for OAMP by construction.** Once `delete_memory(F)` is called, F is gone from the substrate. A separate Audit Vault add-on can preserve it, but the *substrate itself* cannot answer historical queries about superseded claims. For HIPAA-grade amendment workflows where the original record must remain visible alongside the amendment, OAMP requires an external system to be the source of truth — defeating the substrate-of-truth model.

**A3 fails for OAMP for the same reason.** The chain F → A is irrecoverable from the substrate alone once delete has been issued. EBR's `supersedes` pointer + append-only store make the chain free.

**This is the empirical anchor for the LNCS §5 HIPAA argument.** It demonstrates that the difference between OAMP and EBR is not about retrieval quality or latency — both are competitive on A1. The difference is about the *shape of the substrate*: OAMP's mutation semantics make HIPAA amendment workflows architecturally impossible inside the substrate; EBR's append-never-mutate makes them free.

## 10. Extensions to wire in for the published version

Before the notebook ships alongside the paper, two extensions land:

1. **Real-substrate adapters.** Replace `OAMPSubstrate` and `EBRSubstrate` mocks with real adapters:
   - OAMP via `oracleads/oraclesai` Python SDK (when published) OR a faithful re-implementation of `add_memory` / `delete_memory` / `query` against Oracle 23ai vector store
   - EBR via the `accretive-substrate` Node/Go citation API (once the public repo flips per [REVIEW-NOTES.md](https://github.com/ramene/accretive-substrate/blob/main/REVIEW-NOTES.md))
2. **LLM-judge quality.** Add Oracle's third axis — a quality rubric scored by an external judge model (Claude / GPT-4 / Gemini, ideally averaged) on a 1-5 scale per: factual correctness, citation accuracy, audit-trail completeness. Match Oracle's `oracle_agent_memory_benchmarks.ipynb` discipline.

Both extensions slot in without changing the assertion logic — the assertions are about substrate semantics, not implementation quality.

## 11. Citation

If you use this notebook in published work, cite both:

- The Oracle DevRel benchmark this discipline is inherited from: [oracle-devrel/oracle-ai-developer-hub — `oracle_agent_memory_benchmarks.ipynb`](https://github.com/oracle-devrel/oracle-ai-developer-hub) (gather full citation)
- The EBR / memory-oracle papers: position paper `paper/coala-extension/main.tex` and clinical case study `paper/lncs/main.tex` in this repo

**Companion memory file** (for context restoration on session compaction): `~/.claude/projects/-Users-ramene--remote--builds-karve-ai/memory/project_oracle_anthropic_mongodb_synthesis.md`